# Overdose Record Clustering and Model Experiments

Collaborative exploratory analyses of record patterns. Model labels and scores are not individual health-risk estimates or operational predictions.


In [ ]:
#Basic Universal Libraries
import pandas as pd
import numpy as np
import time    # For comparing execution speed of models
import matplotlib.pyplot as plt # For basic visualizations
import seaborn as sns # for advanced visuals
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, mean_squared_error, r2_score

In [ ]:
from pathlib import Path

DATA_PATH = Path("data/accidental_drug_related_deaths.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download the CSV linked in the README."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} records and {len(df.columns)} columns.")


In [ ]:
#Configuring Display (for pycharm)
desired_width=5000
pd.set_option('display.width', desired_width)
np.set_printoptions(linewidth=desired_width)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

#Step 1
#inferring data from our data set
print(df.head(5))
print(df.info())
print(df.shape)  #so we know th number of columns

#Gettting an idea for the new unique values in each feature
for column in df.columns:
    print(f"**Column:** `{column}`")
    value_counts = df[column].value_counts()
    formatted_values = [f"{value} ({count})" for value, count in value_counts.items()]
    print("  Unique Values:", ", ".join(formatted_values))

    print("-" * 40)
print("Shape of final DataFrame is: ", df.shape)
print("Missing Values: \n", df.isna().sum())

In [ ]:
#Removing rows and columns unrelated to research objectives
# Define the allowed values for each column
allowed_date_types = ['Date of death']  # Allow only DofDeath for 'Date Type' (not injury or reported)
allowed_sexes = ['Male', 'Female']
allowed_races = ['White', 'Black', 'Black or African American']
allowed_residence_cities = ['HARTFORD', 'NEW HAVEN', 'WATERBURY', 'BRIDGEPORT', 'NEW BRITAIN', 'BRISTOL', 'MERIDEN', 'NORWICH', 'WEST HAVEN', 'TORRINGTON', 'NEW LONDON', 'DANBURY', 'MANCHESTER', 'EAST HARTFORD', 'MIDDLETOWN', 'STAMFORD', 'MILFORD', 'STRATFORD', 'HAMDEN', 'NORWALK' ]
allowed_death_cities = ['HARTFORD', 'NEW HAVEN', 'WATERBURY', 'BRIDGEPORT', 'NEW BRITAIN', 'BRISTOL', 'MERIDEN', 'NORWICH', 'WEST HAVEN', 'TORRINGTON', 'NEW LONDON', 'DANBURY', 'MANCHESTER', 'EAST HARTFORD', 'MIDDLETOWN', 'STAMFORD', 'MILFORD', 'STRATFORD', 'HAMDEN', 'NORWALK' ]

# Columns to drop
columns_to_drop = ['DeathCityGeo', 'InjuryCityGeo', 'Other', 'Any Opioid',
                   'Heroin/Morph/Codeine', 'Opiate NOS', 'Fentanyl Analogue', 'Heroin death certificate (DC)',
                   'Other Significant Conditions ', 'Cause of Death', 'Manner of Death', 'Location if Other',
                   'Location', 'Death State', 'Death County', 'Description of Injury',
                   'Injury Place', 'Injury State', 'Injury County', 'Injury City',
                   'Residence State', 'Residence County', 'Ethnicity', 'ResidenceCityGeo', 'Other Opioid']

# Drop the columns
refined_df = df.drop(columns=columns_to_drop)

# Filter the DataFrame based on the allowed values
death_df = refined_df[
    refined_df['Date Type'].isin(allowed_date_types) &
    refined_df['Sex'].isin(allowed_sexes) &
    refined_df['Residence City'].isin(allowed_residence_cities) &
    refined_df['Death City'].isin(allowed_death_cities) &
    refined_df['Race'].isin(allowed_races)
]

# Simplify race
# Convert the 'Race' column back to its original string representation before applying the lambda function
death_df['Race'] = death_df['Race'].astype(str)
death_df['Race'] = death_df['Race'].apply(lambda race: 'White' if 'White' in race else ('Black' if 'Black' in race or 'Black or African American' in race else race))


# Extracting the Year from date for better general anlaysis of broad trends

if 'Date' in death_df.columns:
    # Convert 'Date' column to datetime, handling errors
    death_df['Date'] = pd.to_datetime(death_df['Date'], errors='coerce')

    # Extract year and create a new 'Year' column
    death_df['Year'] = death_df['Date'].dt.year

    # Drop the original 'Date' column (optional, if you don't need it anymore)
    death_df = death_df.drop(columns=['Date'])


drug_features = [

    'Heroin', 'Cocaine', 'Fentanyl',

    'Oxycodone', 'Oxymorphone', 'Ethanol',

    'Hydrocodone', 'Benzodiazepine', 'Methadone', 'Meth/Amphetamine',

    'Amphet', 'Tramad', 'Hydromorphone', 'Morphine (Not Heroin)',

    'Xylazine', 'Gabapentin'

]



for col in drug_features:

    death_df[col] = death_df[col].apply(lambda x: 1 if x == 'Y' else 0)

# Reset the index of the new DataFrame so as to avoid index errors
death_df.reset_index(drop=True, inplace=True)

#Gettting an idea for the unique and missing values in each feature
for column in death_df.columns:
    print(f"**Column:** `{column}`")
    value_counts = death_df[column].value_counts()
    formatted_values = [f"{value} ({count})" for value, count in value_counts.items()]
    print("  Unique Values:", ", ".join(formatted_values))

    print("-" * 40)
print("Missing Values: \n", death_df.isna().sum())
print("Shape of final DataFrame is: ", death_df.shape)


In [ ]:
# Save the prepared data for the clustering and modeling sections
from pathlib import Path
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
processed_path = output_dir / "cleaned_overdose_records.csv"
death_df.to_csv(processed_path, index=False)
print(f"Cleaned dataset saved to {processed_path}")


# **Clustering**

Installing and Importing libraries for Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, mutual_info_score
#adjusted_rand_score, adjusted_mutual_info_score, homogeneity_score, completeness_score, v_measure_score
# Cannot use for this objective as requires ground truth
from scipy.spatial.distance import pdist, squareform # For distance matrix calculations
from sklearn.neighbors import NearestNeighbors # For KMeans
#!pip install gower for mixed categorical and binary data #Disregarded due to dominance of binary vals
from sklearn.decomposition import PCA   #For visualizin multidimensional data in 2d
from sklearn.cluster import KMeans # Numerical clustering
from kmodes.kmodes import KModes # Categorical clustering
#from sklearn.cluster import DBSCAN # Density-based clustering approach
import hdbscan # Density-based clustering with anomaly detection and automatic epsilon tuning
#Some sampling tools and lbraries
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
#Preprocessing Techniques
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, OneHotEncoder, KBinsDiscretizer
# For feature importance scores and feature selection
from scipy.stats import chi2_contingency

# Preprocessing for Clustering

# Dropping irrelevant rows, columns and stratification

In [ ]:
from pathlib import Path

PROCESSED_DATA_PATH = Path("outputs/cleaned_overdose_records.csv")
if not PROCESSED_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Processed dataset not found at {PROCESSED_DATA_PATH}. "
        "Run the preprocessing cells in this notebook first."
    )

#Step 0: Defining our metrics
silhouette_all = []  # Stores Silhouette scores for different cluster sizes
SSE = []  # Stores Sum of Squared Errors (SSE) for each cluster count
db_index = []  # Stores Davies-Bouldin index for evaluation
calinski_harabasz = []  # Stores Calinski-Harabasz score

#Step 1: Load the bank dataset
dataset1 = pd.read_csv(PROCESSED_DATA_PATH)
dataset1 = dataset1.drop('Date Type', axis=1)

#Step 2: Clustering-specific Preprocessing
# Filter data before stratification (highlight main cities both smaller and larger)
allowed_death_cities = [ 'NEW HAVEN', 'WATERBURY', 'NORWALK', 'MERIDEN']
dataset1 = dataset1[dataset1['Residence City'].isin(allowed_death_cities) &
                    dataset1['Death City'].isin(allowed_death_cities)]
print(allowed_death_cities)

# 2.1.1: Define the stratification columns (Reduce bias and imbalance while maintaining true variance)
stratification_columns = ['Sex', 'Race']  # Choosing our demographic columns to do stratification on

# 2.1.2 Creating a stratified sample of 1500 rows
num_artificial_points = 15000  # Number of artificial data points to make up
artificial_data = df.sample(n=num_artificial_points, replace=True,  random_state=42)
dataset1_stratified = dataset1.groupby(stratification_columns, group_keys=False).apply(
    lambda x: x.sample(min(len(x), int(12000 * len(x) / len(dataset1))))
)

# Defining allowed features, categories, Discretization and Feature Engineering

In [ ]:
# 2.1.3 Check if the column name exists and is correctly spelled
print(dataset1_stratified.columns)  # Print columns to verify column name

# 2.2.1: Identify categorical and continuous features
categorical_features = dataset1_stratified.select_dtypes(['object', 'bool', 'category']).columns.tolist()
continuous_features = ['Age','Year']
boolean_features = ['Benzodiazepine', 'Gabapentin', 'Oxycodone' ]  # Add your boolean columns here

# 2.2.2: Remove 'Residence City' if it's in the list
if 'Death City' in categorical_features:
    categorical_features.remove('Death City')


# Scaling before discretization, non since discretization is done
#scaler = MinMaxScaler() # Or StandardScaler() if you prefer
#dataset1_stratified[continuous_features] = scaler.fit_transform(dataset1_stratified[continuous_features])

# 2.2.3: Store original year values in a new column
dataset1_stratified['Original_Year'] = dataset1_stratified['Year']

# 2.2.4: Discretize continuous features
discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')  # Adjust n_bins as needed
dataset1_stratified[continuous_features] = discretizer.fit_transform(dataset1_stratified[continuous_features]) #Calling the fit method before accessing bin_edges_

# 2.2.5: Print the range of continuous values for each encoded number
for feature in continuous_features:
    print(f"\nFeature: {feature}")
    for i in range(discretizer.n_bins):
        bin_edges = discretizer.bin_edges_[continuous_features.index(feature)]
        if i < len(bin_edges) - 1:  # Check to prevent index out of bounds
            print(f"Encoded Number {i}: Range [{bin_edges[i]:.2f}, {bin_edges[i + 1]:.2f})")
        else:
            print(f"Encoded Number {i}: Range [{bin_edges[-1]:.2f}, inf)")

# 2.3.1: Create "Time Since First Case" Feature
dataset1_stratified['Years_Since_2012'] = dataset1_stratified['Original_Year'] - dataset1_stratified['Original_Year'].min()

# 2.3.2: Aggregated Drug Usage Features
dataset1_stratified['Opioid Score'] = dataset1_stratified[['Hydromorphone', 'Morphine (Not Heroin)', 'Fentanyl', 'Oxycodone']].sum(axis=1)
dataset1_stratified['Poly-drug Score'] = dataset1_stratified[['Heroin',
       'Cocaine', 'Fentanyl', 'Oxycodone', 'Oxymorphone', 'Ethanol',
       'Hydrocodone', 'Benzodiazepine', 'Methadone', 'Meth/Amphetamine',
       'Amphet', 'Tramad', 'Hydromorphone', 'Morphine (Not Heroin)',
       'Xylazine', 'Gabapentin']].astype(bool).sum(axis=1)
dataset1_stratified['Sedative-Opioid Interaction'] = (dataset1_stratified['Benzodiazepine'] & dataset1_stratified[['Hydromorphone', 'Morphine (Not Heroin)', 'Fentanyl', 'Oxycodone']].any(axis=1)).astype(int)

# 2.3.3: 'Urbanization' feature
# Assuming cities with populations > 50,000 are urban
# Create a list of urban cities based on general knowledge and dataset observation
urban_cities = ['HARTFORD', 'BRIDGEPORT', 'NEW HAVEN', 'WATERBURY', 'NEW BRITAIN', 'DANBURY', 'NORWALK', 'WEST HAVEN', 'STAMFORD', 'MERIDEN']
dataset1_stratified['Urbanization'] = dataset1_stratified['Residence City'].apply(lambda city: 1 if city in urban_cities else 0)

# 2.3.4: 'City Risk' feature
# Calculate the frequency of each Death City
city_frequencies = dataset1_stratified['Death City'].value_counts()

# Define a threshold for high-risk cities (e.g., top 20% most frequent)
high_risk_threshold = city_frequencies.quantile(0.8)

# Create the City Risk feature
dataset1_stratified['City Risk'] = dataset1_stratified['Death City'].apply(lambda city: 1 if city_frequencies[city] >= high_risk_threshold else 0)


# 2.4.1 : Identify categorical and continuous features
# Updated to include all relevant features:
categorical_features = dataset1_stratified.select_dtypes(['object', 'bool', 'category']).columns.tolist()
continuous_features = ['Year', 'Age']
boolean_features = ['Heroin',
       'Cocaine', 'Fentanyl', 'Oxycodone', 'Oxymorphone', 'Ethanol',
       'Hydrocodone', 'Benzodiazepine', 'Methadone', 'Meth/Amphetamine',
       'Amphet', 'Tramad', 'Hydromorphone', 'Morphine (Not Heroin)',
       'Xylazine', 'Gabapentin' ]  # Add your boolean columns herenew_feature = ['Opioid Score'] #Including the new engineered features
new_feature = ['Poly-drug Score', 'City Risk']       # 'Sedative-Opioid Interaction',  #'Urbanization'
all_features = categorical_features + new_feature + boolean_features + continuous_features
print(all_features)

# 2.4.2: Low contribution to clusters
all_features.remove('Age')
all_features.remove('Year')
all_features.remove('Sex')
for item in [ 'Cocaine', 'Fentanyl', 'Gabapentin', 'Oxycodone', 'Heroin',  'Oxymorphone', 'Ethanol', 'Hydrocodone', 'Methadone', 'Meth/Amphetamine', 'Amphet', 'Tramad', 'Hydromorphone', 'Morphine (Not Heroin)', 'Xylazine']:
    all_features.remove(item) # remove each item individually in a loop

# 2.4.3: printing to ensure code works as intended

print("\nCategorical Features:", categorical_features)
print("Continuous Features:", continuous_features)
print("Boolean Features:", boolean_features)
print("New Features:", new_feature)
print("All Features: ", all_features)
print()


##### COMMENTED OUT AS IT REDUCED MEANINGFULNESS OF OUTPUT #######

# Group infrequent categories in 'Residence City' into 'Other'
#category_frequencies = dataset1_stratified['Residence City'].value_counts(normalize=True)
#infrequent_categories = category_frequencies[category_frequencies < 0.05].index
#dataset1_stratified['Residence City'] = dataset1_stratified['Residence City'].apply(lambda x: 'Other' if x in infrequent_categories else x)

# Group infrequent categories in 'Death City' into 'Other'
#category_frequencies = dataset1_stratified['Death City'].value_counts(normalize=True)
#infrequent_categories = category_frequencies[category_frequencies < 0.05].index
#dataset1_stratified['Death City'] = dataset1_stratified['Death City'].apply(lambda x: 'Other' if x in infrequent_categories else x)

# Copying the Dataset, Encoding, and Maximizing Variance. Initializes evaluation metrics

In [ ]:
# 2.5: copy
dataset = dataset1_stratified[all_features].copy()

# 2.6.1 : Encode
le = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # sparse=False for dense output, handle_unknown to ignore new categories in test
X_2 = le.fit_transform(dataset) # Directly applying fit_transform on the DataFrame

# 2.6.2 : Convert the encoded data back into a DataFrame for compatibility with KModes
feature_names = le.get_feature_names_out(input_features=dataset.columns)
X_2 = pd.DataFrame(X_2, columns=feature_names, index=dataset.index)

# 2.6.3 : Remove the features from X_2 (To maximize differences between low and high variety users)
X_2 = X_2.drop(columns=['Poly-drug Score_3', 'Poly-drug Score_4'])

# 2.6.4 : Initialize evaluation metrics to Kmodes clustering
cost = [] # Stores clustering cost for different cluster size
silhouette_all = []


# **Clustering Model 1 - KMeans**

In [ ]:
# --- 3. KMeans Clustering ---
# 3.1: Calculate Hamming distance matrix (if not already done)
distance_matrix = squareform(pdist(X_2, metric='hamming'))  # Use Hamming distance

# 3.2: Record start time
start_time_kmeans = time.time()

# 3.3: Perform KMeans clustering for different cluster sizes (2 to 6)
wcss = []
silhouette_scores = []
k_range = range(2, 7)  # Same range as KModes for comparison
for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=20, max_iter=300, random_state=42)
    labels = kmeans.fit_predict(distance_matrix)
    wcss.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(distance_matrix, labels, metric='precomputed'))

# 3.4: Record end time and Calculate execution time
end_time_kmeans = time.time()
execution_time_kmeans = end_time_kmeans - start_time_kmeans
print(f"\nKMeans Execution Time: {execution_time_kmeans:.4f} seconds\n")

# 3.5: Plotting WCSS (Elbow Method)
plt.figure(figsize=(10, 5))
plt.plot(k_range, wcss, marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method for Optimal K (KMeans)")  # Added (KMeans) to the title
plt.show()

# 3.6: Plotting Silhouette Score
plt.figure(figsize=(10, 5))
plt.plot(k_range, silhouette_scores, marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs. Number of Clusters (KMeans)")  # Added (KMeans) to the title
plt.show()

# 3.7: Choosing the best K based on silhouette score
best_k_kmeans = k_range[np.argmax(silhouette_scores)]
print("Best K for KMeans:", best_k_kmeans)

# 3.8: KMeans Clustering with the chosen K
kmeans = KMeans(n_clusters=best_k_kmeans, random_state=42)
kmeans_labels = kmeans.fit_predict(distance_matrix)

# 3.9: Evaluation Metrics for KMeans
print("KMeans Silhouette Score:", silhouette_score(distance_matrix, kmeans_labels, metric='precomputed'))
print("KMeans Davies-Bouldin Score:", davies_bouldin_score(distance_matrix, kmeans_labels))
print("KMeans Calinski-Harabasz Score:", calinski_harabasz_score(distance_matrix, kmeans_labels))


# 3.10: Visualizing Clusters for KMeans using PCA for dimensionality reduction
scaled_distance_matrix = distance_matrix * 1
pca = PCA(n_components=2)  # Reduce to 2 dimensions for plotting
pca_data = pca.fit_transform(scaled_distance_matrix)

plt.figure(figsize=(3, 3))
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=kmeans_labels, palette='viridis')
plt.title("KMeans Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

# 3.11: Assuming kmeans is your fitted KMeans model
centroids = kmeans.cluster_centers_
feature_importances_kmeans = np.var(centroids, axis=0)

# 3.12: Print feature importances
for i, feature in enumerate(feature_names):  # feature_names from your OneHotEncoder
    print(f"Feature '{feature}': Importance = {feature_importances_kmeans[i]:.4f}")

# **Clustering Model 2 - HDBSCAN**

In [ ]:
# --- 4. HDBSCAN (DENSITY BASED) Clustering ---
# 4.1: Normalize the distance matrix (if not already done)
normalized_distance_matrix = scaled_distance_matrix / np.max(scaled_distance_matrix)

# 4.2: K-Distance Graph (Optional, for understanding structure)
nn = NearestNeighbors(n_neighbors=7, metric='precomputed')  # Adjust n_neighbors if needed
neighbor_distances, _ = nn.fit(normalized_distance_matrix).kneighbors(normalized_distance_matrix)
k_distances = np.sort(neighbor_distances[:, -1])

plt.figure(figsize=(10, 5))
plt.plot(k_distances)
plt.xlabel("Points sorted by distance")
plt.ylabel("5th Nearest Neighbor Distance")  # Update if n_neighbors is changed
plt.title("K-Distance Graph for DBSCAN/HDBSCAN")
plt.show()

# 4.3: Perform HDBSCAN clustering
print("Running HDBSCAN...")

# 4.4: Record start time
start_time_dbscan = time.time()

# 4.5: Convert distance matrix to float64 if needed
normalized_distance_matrix = normalized_distance_matrix.astype(np.float64)

hdbscan_clusterer = hdbscan.HDBSCAN(
    min_cluster_size=130,  # Matches min_samples in DBSCAN
    metric='precomputed',
    cluster_selection_method='eom',  # Default: 'eom' (to havemore stable), 'leaf' (for finely drawn clusters)
    alpha=0.95,
    cluster_selection_epsilon=0.950
)

hdbscan_labels = hdbscan_clusterer.fit_predict(normalized_distance_matrix)

# 4.6: Record end time and Calculate execution time
end_time_dbscan = time.time()
execution_time_dbscan = end_time_dbscan - start_time_dbscan
print(f"\nDBSCAN Execution Time: {execution_time_dbscan:.4f} seconds\n")

# 4.7: Evaluation Metrics for HDBSCAN (Ignoring Noise)
valid_labels = hdbscan_labels != -1
if np.any(valid_labels):
    print("HDBSCAN Silhouette Score:", silhouette_score(
        normalized_distance_matrix[valid_labels, :][:, valid_labels],
        hdbscan_labels[valid_labels],
        metric='precomputed'
    ))
    print("HDBSCAN Davies-Bouldin Score:", davies_bouldin_score(
        X_2[valid_labels], hdbscan_labels[valid_labels]  # Use original data for DBI
    ))
    print("HDBSCAN Calinski-Harabasz Score:", calinski_harabasz_score(
        X_2[valid_labels], hdbscan_labels[valid_labels]  # Use original data for CHI
    ))
else:
    print("HDBSCAN did not find valid clusters.")

# 4.8: Visualizing Clusters for HDBSCAN with some PCA for reducing to 2d
pca = PCA(n_components=2)  # Reduce dimensions to plot easy
pca_data = pca.fit_transform(normalized_distance_matrix)  # Using normalized distance matrix

plt.figure(figsize=(3, 3))
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=hdbscan_labels, palette='viridis')
plt.title("HDBSCAN Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()



# Clustering (Density) Based Outlier Detection

In [ ]:
# CLUSTERING BASED OUTLIER DETECTION
# 5.1 Get outlier scores:
outlier_scores = hdbscan_clusterer.outlier_scores_

# 5.2 Identify potential outliers (noise points):
outlier_indices = np.where(hdbscan_labels == -1)[0]

# 5.3 Get the outlier data:
outlier_data = dataset1_stratified.iloc[outlier_indices]  # Access original data

# 5.4 Print basic statistics:
print("\nOutlier Statistics:")
print(outlier_data.describe())  # Basic descriptive statistics

# 5.5 Print outlier scores (optional):
print("\nOutlier Scores:")
for i, score in zip(outlier_indices, outlier_scores[outlier_indices]):
    print(f"Data point index {i}: Outlier score = {score:.4f}")

# 5.6 Explore specific features for outliers:
# Example: Print the distribution of 'Age' for outliers
print("\nDistribution of 'Age' for Outliers:")
print(outlier_data['Age'].value_counts())

# 5.7 Compare outlier statistics to the overall dataset statistics:
print("\nOverall Dataset Statistics:")
print(dataset1_stratified.describe())  # Statistics for the entire dataset

# **Clustering Model 3 - KModes**

In [ ]:
# --- 6. KModes (CATEGORICAL) Clustering ---
# 6.1 Record start time
start_time_kmodes = time.time()

# 6.2 Perform KModes clustering for different cluster sizes (2 to 12)
# Not using n_init or max_iter because of time complexity optimization
K = range(2, 7)
for num_clusters in K:
    print(f"Fitting KModes with K = {num_clusters}")
    kmode = KModes(n_clusters=num_clusters, init="Huang", n_init=10, random_state=42)  #Initialize KModes
    labels = kmode.fit_predict(dataset)  #Fit the model and get cluster labels # changed to fit_predict

    # 6.2.2 Store silhouette score
    silhouette_all.append(silhouette_score(X_2, labels, metric='hamming'))
    db_index.append(davies_bouldin_score(X_2, labels))
    calinski_harabasz.append(calinski_harabasz_score(X_2, labels))

    # 6.2.3 Append the cost for the current cluster size to the cost list
    cost.append(kmode.cost_)  # Store clustering cost


# 6.3 Find the best K (highest Silhouette score)
best_k = K[silhouette_all.index(max(silhouette_all))]

final_kmode = KModes(n_clusters=best_k, init="Huang", random_state=42)
final_labels = final_kmode.fit_predict(X_2)

# 6.4 Record end time and Calculate execution time
end_time_kmodes = time.time()
execution_time_kmodes = end_time_kmodes - start_time_kmodes
print(f"\nKModes Execution Time: {execution_time_kmodes:.4f} seconds\n")

# Plotting Evaluation metrics and Mutual Information scores (for model tuning)

In [ ]:
# 6.5 Plot Cost Function (Elbow Method for KModes)
plt.figure(figsize=(12, 6))
plt.plot(K, cost, 'bx-')
plt.xlabel('No. of clusters')
plt.ylabel('Cost')
plt.title('Elbow Method for KModes Clustering')
plt.show()

# 6.6 Plot Silhouette Scores for KModes Clustering
plt.figure(figsize=(12, 6))
plt.plot(K, silhouette_all, 'bx-')
plt.xlabel('No. of clusters')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for KModes')

# 6.7 Highlight the best K with a vertical line
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K ({best_k})')  # Add vertical line
plt.legend()  # Show legend

plt.show()

# 6.8 Evaluation Metrics for KMeans
print("Shape of K:", np.array(K).shape) # Print shape of K
print("Shape of X_2:", X_2.shape) # Print shape of X_2
print("Shape of labels:", labels.shape) # Print shape of labels
print("\nKModes Silhouette Score:", silhouette_score(X_2, final_labels, metric='hamming')) # Changed K to X_2
print("KModes Davies-Bouldin Score:", davies_bouldin_score(X_2, final_labels)) # Changed K to X_2
print("KModes Calinski-Harabasz Score:", calinski_harabasz_score(X_2, final_labels)) # Changed K to X_2
print("\nFinal cluster labels: ", np.unique(final_labels))
print("Best K value", best_k)



# 6.9 Plot Davies-Bouldin Index (Lower is Better)
plt.figure(figsize=(12, 6))
plt.plot(K, db_index, marker='o', linestyle='-', color='red')
plt.xlabel('Number of Clusters')
plt.ylabel('Davies-Bouldin Index')
plt.title('Davies-Bouldin Index for Different Cluster Numbers (Lower is Better)')
plt.show()

# 6.10 Plot Calinski-Harabasz Score (Higher is Better)
plt.figure(figsize=(12, 6))
plt.plot(K, calinski_harabasz, marker='o', linestyle='-', color='blue')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Calinski-Harabasz Score')
plt.title('Calinski-Harabasz Score for Different Cluster Numbers (Higher is Better)')
plt.show()

for feature in all_features:
    contingency_table = pd.crosstab(dataset[feature], final_labels)
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"Feature '{feature}': Chi-squared = {chi2:.4f}, p-value = {p:.4f}")
print()

# 6.11 Calculate and print mutual information for each feature

print('Mutual Information for all features to inform feature selection.\n')
for feature in all_features:
    # KMeans
    mi_kmeans = mutual_info_score(dataset[feature], kmeans_labels)
    print(f"Feature '{feature}': Mutual Information (KMeans) = {mi_kmeans:.4f}")

    # KModes
    mi_kmodes = mutual_info_score(dataset[feature], final_labels)  # Assuming final_labels are from KModes
    print(f"Feature '{feature}': Mutual Information (KModes) = {mi_kmodes:.4f}")

    # HDBSCAN (considering noise points)
    valid_labels = hdbscan_labels != -1
    if np.any(valid_labels):
        mi_hdbscan = mutual_info_score(dataset[feature][valid_labels], hdbscan_labels[valid_labels])
        print(f"Feature '{feature}': Mutual Information (HDBSCAN) = {mi_hdbscan:.4f}")
    else:
        print(f"Feature '{feature}': Mutual Information (HDBSCAN) = N/A (All noise)")

    print()  # Add a newline for better readability

# If kmode is the fitted KModes object then
#for cluster in range(kmode.n_clusters):
 #   print(f"Cluster {cluster}:")
  #  for i, feature in enumerate(dataset.columns):  # dataset is your original DataFrame
   #     print(f"  {feature}: {kmode.cluster_centroids_[cluster, i]}")


# Visualizing KModes by Principal Component Analysis

In [ ]:
# 6.12 Calculate Hamming distance matrix
distance_matrix = squareform(pdist(X_2, metric='hamming'))  # Use Hamming distance

# 6.13 Visualizing the Clusters for KModes using PCA
pca = PCA(n_components=2)  # Reduces to 2d for plots
pca_data = pca.fit_transform(distance_matrix)

plt.figure(figsize=(3, 3))
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=final_labels, palette='viridis')  # Use 'labels' from KModes
plt.title("KModes Clustering")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

In [ ]:
# 6.14 Perform final KModes clustering with the best K (chosen as 6)
kmode = KModes(n_clusters=best_k, random_state=42)
labels = kmode.fit(dataset).labels_

# 6.15 Print cluster centroids for interpretation
for c in kmode.cluster_centroids_:
    print(c)

# 6.17 Print final cluster labels
print("Cluster labels:", labels)

# 6.18 Plot Cluster Distribution
# Helps check for imbalanced clusters
plt.figure(figsize=(8, 5))
sns.countplot(x=labels)
plt.xlabel('Cluster Number')
plt.ylabel('Number of Data Points')
plt.title('Distribution of Data Points in Clusters (KModes)')
plt.show()

'''# 6.19 Analyze the Most frequent Features in Each Cluster
dataset['Cluster'] = labels  # Add cluster labels to dataset

for cluster in range(best_k):
    print(f"\nCluster {cluster} Characteristics:")
    # Use dataset (which has the 'Cluster' column) to filter
    cluster_data = dataset[dataset['Cluster'] == cluster]
    # Then use dataset1_stratified (with all features) to get mode values
    cluster_data_stratified = dataset1_stratified.loc[cluster_data.index]  # Use cluster_data index to filter

    if not cluster_data_stratified.empty:
        mode_values = cluster_data_stratified.mode().iloc[0]  # Get mode for all features
        for feature in dataset1_stratified.columns:  # Iterate through all features
            if feature in mode_values:  # Check if feature has a mode value
                print(f"{feature}: {mode_values[feature]}")  # Print feature and its mode
            else:
                print(f"{feature}: No Majority Category")  # Print if no majority category
    else:
        print(f"No data points found for cluster {cluster}")

# 6.20 Plot Pairplot for Categorical Data
plt.figure(figsize=(12, 6))

# Check if 'Race' is in the dataset columns
if 'Race' in dataset.columns:
    # Find the index of 'Race' in categorical_features
    # race_index = categorical_features.index('Race')
    race_index = dataset.columns.get_loc('Race')  # Use dataset.columns to get the location
    # Use the race_index to select the 'Race' column
    sns.heatmap(pd.crosstab(dataset['Cluster'], dataset['Race']),  # Directly use 'Race' column
                cmap="coolwarm", annot=True)

    plt.title("Heatmap of Race Feature by Cluster")
    plt.xlabel("Feature Values")
    plt.ylabel("Cluster Number")
    plt.show()
else:
    print("Column 'Race' not found in the dataset.")'''

# Main Visualizations and Insights for Clustering

**KModes - Insights**

**Cluster Features across all 3 Models**

In [ ]:
# --- 7. MAIN VISUALIZATION OF OUTPUT ---
# 7.1 Function to plot frequency distributions
def plot_cluster_distributions(data, cluster_labels, cluster_type):
    """Plots frequency distributions of various features for each cluster."""
    features = ['Race', 'Age', 'Sex', 'Residence City', 'Death City',
                'Original_Year', 'Opioid Score', 'Poly-drug Score',
                'Sedative-Opioid Interaction']  # Features to plot
    num_clusters = len(np.unique(cluster_labels))
    # 7.1.1 Adjust figure size dynamically based on number of clusters and features
    fig, axes = plt.subplots(len(features), num_clusters, figsize=(30, 24))
    fig.suptitle(f"Cluster Distributions ({cluster_type})", fontsize=28, fontweight='bold')
    palette = sns.color_palette("Set2")
    for i, feature in enumerate(features):
        for j, cluster in enumerate(np.unique(cluster_labels)):
            cluster_data = data[cluster_labels == cluster]

            ax = axes[i, j] if num_clusters > 1 else axes[i]
            sns.countplot(x=feature, data=cluster_data, ax=ax, hue=feature, palette=sns.color_palette("Set2", n_colors=cluster_data[feature].nunique()))
            handles, labels = axes[i, j].get_legend_handles_labels()
            if labels:  # Only add legend if there are labels
              axes[i, j].legend(title=feature, bbox_to_anchor=(1.05, 1), loc='upper left')

            ax.set_title(f"Cluster {cluster}", fontsize=18, fontweight='bold', pad=15)
            ax.set_xlabel(feature, fontsize=14)
            ax.set_ylabel("Frequency", fontsize=14)
            ax.tick_params(axis='x', rotation=45, labelsize=12)  # Final optimal rotation
            ax.tick_params(axis='y', labelsize=12)

            # 7.1.2 Annotate bars with their counts
            for p in ax.patches:
                height = p.get_height()
                if height > 0:  # Only annotate if height is greater than 0
                    ax.annotate(f'{height}', (p.get_x() + p.get_width() / 2., height),
                                ha='center', va='bottom', fontsize=10)
    # Removed extra plt.figure call (it was creating a new blank figure)
    plt.figure(figsize=(70, 30))
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.subplots_adjust(hspace=2.5, wspace=0.8)  # Maximized spacing to remove clutter
    plt.show()
# --- KMeans ---
plot_cluster_distributions(dataset1_stratified, kmeans_labels, "KMeans")

# --- KModes ---
plot_cluster_distributions(dataset1_stratified, final_labels, "KModes")

# --- HDBSCAN ---
valid_labels = hdbscan_labels != -1
plot_cluster_distributions(dataset1_stratified[valid_labels], hdbscan_labels[valid_labels], "HDBSCAN")


# Main Visualizations and Insights for Clustering

**KModes - Insights**

**Cluster Features across all 3 Models**

**Visualizing and Comparing Evaluation Metrics**

In [ ]:
# 7.2 Calculate evaluation metrics ---
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

# 7.2.1 KMeans
silhouette_kmeans = silhouette_score(distance_matrix, kmeans_labels, metric='precomputed')
davies_bouldin_kmeans = davies_bouldin_score(distance_matrix, kmeans_labels)
calinski_harabasz_kmeans = calinski_harabasz_score(distance_matrix, kmeans_labels)

# 7.2.2 KModes
silhouette_kmodes = silhouette_score(X_2, final_labels, metric='hamming')
davies_bouldin_kmodes = davies_bouldin_score(X_2, final_labels)
calinski_harabasz_kmodes = calinski_harabasz_score(X_2, final_labels)

# 7.2.3 HDBSCAN (considering noise points)
valid_labels = hdbscan_labels != -1
if np.any(valid_labels):
    silhouette_hdbscan = silhouette_score(normalized_distance_matrix[valid_labels, :][:, valid_labels], hdbscan_labels[valid_labels], metric='precomputed')
    davies_bouldin_hdbscan = davies_bouldin_score(X_2[valid_labels], hdbscan_labels[valid_labels])
    calinski_harabasz_hdbscan = calinski_harabasz_score(X_2[valid_labels], hdbscan_labels[valid_labels])
else:
    silhouette_hdbscan, davies_bouldin_hdbscan, calinski_harabasz_hdbscan = np.nan, np.nan, np.nan  # Assign NaN if no valid clusters

# --- Create a Pandas DataFrame for the visualization ---
df_vis = pd.DataFrame({
    'Model': ['KMeans', 'KModes', 'HDBSCAN'] * 3,
    'Metric': ['Silhouette'] * 3 + ['Davies-Bouldin'] * 3 + ['Calinski-Harabasz'] * 3,
    'Score': [silhouette_kmeans, silhouette_kmodes, silhouette_hdbscan,
              davies_bouldin_kmeans, davies_bouldin_kmodes, davies_bouldin_hdbscan,
              calinski_harabasz_kmeans, calinski_harabasz_kmodes, calinski_harabasz_hdbscan]
})

# --- 7.2.4 Create separate visualizations with hue and adjusted ranges ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- 7.2.5 Silhouette Score ---
ax = sns.barplot(x='Metric', y='Score', hue='Model', data=df_vis[df_vis['Metric'] == 'Silhouette'], ax=axes[0])
axes[0].set_ylabel('Silhouette Score (Higher is Better)')
axes[0].set_title('Silhouette Score')
axes[0].set_ylim(0.3, 0.8)  # Limit y-axis range to 0-1

# 7.2.6 Add annotations to Silhouette Score bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 10),  # Adjust vertical offset as needed
                textcoords='offset points')

# --- 7.2.7 Davies-Bouldin Index ---
ax = sns.barplot(x='Metric', y='Score', hue='Model', data=df_vis[df_vis['Metric'] == 'Davies-Bouldin'], ax=axes[1], palette='viridis')
axes[1].set_ylabel('Davies-Bouldin Index (Lower is Better)')
axes[1].set_title('Davies-Bouldin Index')
axes[1].set_ylim(0.8, 1.4)  # Center y-axis around 1 (0.8-1.2)
axes[1].invert_yaxis()  # Invert y-axis for DBI

# 7.2.8 Add annotations to Davies-Bouldin Index bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, -10),  # Adjust vertical offset as needed
                textcoords='offset points')

# --- 7.2.9 Calinski-Harabasz Index ---
ax = sns.barplot(x='Metric', y='Score', hue='Model', data=df_vis[df_vis['Metric'] == 'Calinski-Harabasz'], ax=axes[2], palette="colorblind")
axes[2].set_ylabel('Calinski-Harabasz Index (Higher is Better)')
axes[2].set_title('Calinski-Harabasz Index')

# 7.2.10 Add annotations to Calinski-Harabasz Index bars
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center',
                xytext=(0, 10),  # Adjust vertical offset as needed
                textcoords='offset points')

# 7.2.11 Add legends and adjust layout
for ax in axes:
    ax.legend(title='Model')  # Add legend with title

fig.tight_layout()
plt.show()

**Checking Demographic features Race and Sex Distribution in Clusters in all models**

In [ ]:
# 7.3 Checking Demographics for main model
features = ['Sex', 'Race']
cluster_data = dataset1_stratified  # using the full dataset

for feature in features:
    for model_name, cluster_labels in [("KMeans", kmeans_labels), ("KModes", final_labels), ("HDBSCAN", hdbscan_labels)]:
        # 7.3.1 Filter out noise points (if any)
        valid_indices = cluster_labels != -1
        filtered_cluster_labels = cluster_labels[valid_indices]
        filtered_data = cluster_data[valid_indices]

        # 7.3.2  If cluster_labels is a NumPy array, convert it to a Pandas Series
        if isinstance(filtered_cluster_labels, np.ndarray):
            filtered_cluster_labels = pd.Series(filtered_cluster_labels, index=filtered_data.index)

        # 7.3.3  Create a cross-tabulation of Cluster and the feature
        cross_tab = pd.crosstab(filtered_cluster_labels, filtered_data[feature], normalize='index')

        # 7.3.4 Plot the heatmap
        plt.figure(figsize=(8, 6))
        sns.heatmap(cross_tab, annot=True, cmap="YlGnBu", fmt=".2f")
        plt.title(f"{model_name} Clustering: {feature} Distribution per Cluster")
        plt.xlabel("Feature Values")
        plt.ylabel("Cluster Number")
        plt.show()

**Overview of each Race and Sex in Kmeans Clusters**

In [ ]:
# --- 7.4 Heatmap for Sex * Race (KMeans) ---
# Create a new column combining Sex and Race
cluster_data['Sex_Race'] = cluster_data['Sex'] + ' - ' + cluster_data['Race']

# Filter out noise points for KMeans
valid_indices_kmeans = kmeans_labels != -1
filtered_cluster_labels_kmeans = kmeans_labels[valid_indices_kmeans]
filtered_data_kmeans = cluster_data[valid_indices_kmeans]

# Create cross-tabulation for Sex * Race
cross_tab_sex_race = pd.crosstab(
    filtered_cluster_labels_kmeans,
    filtered_data_kmeans['Sex_Race'],
    normalize='index'
)

# Plot the heatmap for Sex * Race (KMeans)
plt.figure(figsize=(10, 8))  # Adjust figure size as needed
sns.heatmap(cross_tab_sex_race, annot=True, cmap="YlGnBu", fmt=".2f")
plt.title("KMeans Clustering: Sex and Race Distribution per Cluster")
plt.xlabel("Sex - Race")
plt.ylabel("Cluster Number")
plt.show()

**Drug related insights for KMeans Cluster**

In [ ]:
# --- 7.5 Frequency Plots for KMeans Clusters ---
# --- Grouped Bar Chart for Drug-related Columns ---
drug_related_columns = ['Heroin', 'Cocaine', 'Fentanyl', 'Oxycodone', 'Oxymorphone',
                        'Ethanol', 'Hydrocodone', 'Benzodiazepine', 'Methadone',
                        'Meth/Amphetamine', 'Amphet', 'Tramad', 'Hydromorphone',
                        'Morphine (Not Heroin)', 'Xylazine', 'Gabapentin']
num_clusters = len(np.unique(kmeans_labels))

# Create a DataFrame for plotting
cluster_frequencies = []
for cluster in range(num_clusters):
    cluster_data = dataset1_stratified[kmeans_labels == cluster]
    frequencies = cluster_data[drug_related_columns].mean()  # Mean for boolean (1/0) is frequency of 1
    frequencies['Cluster'] = cluster  # Add cluster number
    cluster_frequencies.append(frequencies)

cluster_frequencies_df = pd.DataFrame(cluster_frequencies)
cluster_frequencies_df = cluster_frequencies_df.melt(id_vars=['Cluster'], var_name='Drug', value_name='Frequency')

# Plotting
plt.figure(figsize=(20, 6))  # Adjust figure size as needed

# Using a more contrasting color palette
colors = ['#e6194b', '#3cb44b', '#ffe119', '#4363d8', '#f58231', '#911eb4', '#46f0f0', '#f032e6', '#bcf60c', '#fabebe', '#008080', '#e6beff', '#9a6324', '#fffac8', '#800000', '#aaffc3', '#808000', '#ffd8b1', '#000075', '#808080', '#ffffff', '#000000']  # Add more colors if needed
ax = sns.barplot(x='Drug', y='Frequency', hue='Cluster', data=cluster_frequencies_df, palette=colors)

plt.title("Frequency of Drug Usage in KMeans Clusters", fontsize=16, fontweight='bold')
plt.xlabel("Drug", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)  # Rotate x-axis labels for better readability
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')

# Annotate bars with their counts
for p in ax.patches:
    height = p.get_height()
    if height > 0:  # Only annotate if height is greater than 0
        ax.annotate(f'{height:.2f}', (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


### **Distribution of Cities (of Death and Residence)**

In [ ]:
# 7.6 Define a more visually appealing color palette
colors = sns.color_palette("tab10", n_colors=10)  # Generates distinct colors

# --- Function to Plot Cluster Frequencies ---
def plot_cluster_frequencies(data, column_name, title):
    num_clusters = len(np.unique(kmeans_labels))
    fig, ax = plt.subplots(figsize=(14, 6))
    fig.suptitle(title, fontsize=18, fontweight='bold')

    cluster_frequencies = []
    for cluster in range(num_clusters):
        cluster_data = data[kmeans_labels == cluster]
        frequency = cluster_data[column_name].value_counts(normalize=True)
        cluster_frequencies.append(frequency)

    # Convert to DataFrame & fill missing values
    cluster_frequencies_df = pd.DataFrame(cluster_frequencies).fillna(0)
    cluster_frequencies_df.index = range(num_clusters)

    # Create stacked bar plot
    cluster_frequencies_df.plot(
        kind='bar', stacked=True, ax=ax, color=colors[:len(cluster_frequencies_df.columns)]
    )

    # Labels and formatting
    ax.set_xlabel("Cluster Number", fontsize=14)
    ax.set_ylabel("Frequency", fontsize=14)
    ax.set_xticks(np.arange(num_clusters))
    ax.set_xticklabels(np.arange(num_clusters), fontsize=12)
    plt.xticks(rotation=45)  # Rotate x-axis labels for better spacing


    ax.legend(title=column_name, bbox_to_anchor=(1.05, 1), loc='upper left')

    # Annotate bars with percentage values
    for i, cluster in enumerate(cluster_frequencies_df.index):
        cumulative_sum = 0  # To track height positioning for stacked bars
        for j, city in enumerate(cluster_frequencies_df.columns):
            value = cluster_frequencies_df.loc[cluster, city]
            if value > 0:
                ax.text(
                    i, cumulative_sum + value / 2, f'{value:.2%}',
                    ha='center', va='center', color='black', fontsize=10, fontweight='bold'
                )
                cumulative_sum += value  # Update height position for next annotation

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Plot Death City and Residence City
plot_cluster_frequencies(dataset1_stratified, 'Death City', "Frequency of Death City in KMeans Clusters")
plot_cluster_frequencies(dataset1_stratified, 'Residence City', "Frequency of Residence City in KMeans Clusters")


**Decoding**

Feature: Age
* Encoded Number 0: Range [17.00, 34.00)
* Encoded Number 1: Range [34.00, 42.00)
* Encoded Number 2: Range [42.00, 49.00)
* Encoded Number 3: Range [49.00, 57.00)
* Encoded Number 4: Range [57.00, 87.00)

Feature: Year
* Encoded Number 0: Range [2012.00, 2016.00)
* Encoded Number 1: Range [2016.00, 2018.00)
* Encoded Number 2: Range [2018.00, 2020.00)
* Encoded Number 3: Range [2020.00, 2021.00)
* Encoded Number 4: Range [2021.00, inf)

Feature: Poly-drug score
* Number of Drugs Taken
Feature: Sedative-Opioid Interaction
* Encoded Number 0: Not taking both Sedatives (Benzodiazepine) together with Opiods (Fentanyl, Oxycodone, Hydromorphone, Morphine, Heroin, Hydrocodone, Oxymorphone, Methadone, Tramad)
* Encoded Number 1: Taking both Sedatives and Opiods together (Greater risk of respiratory depression, extreme sedation, overdose, addiction and dependance)

Feature: Urbanization
* Encoded Number 0: Relatively Rural
* Encoded Number 1: Relatively Urban
Feature: City Risk
* Encoded Number 0: In city with low risk of drug-related death
* Encoded Number 1: In city with high risk of drug-related death





# **Most Common Values for each feature in each Cluster for all Models**

In [ ]:
# --- 8. Printing Modal values for every feature in each cluster (for all 3 models) ---
dataset['Cluster_KModes'] = labels  # Add KModes cluster labels to dataset

print("\n--- KModes Clustering Results ---\n")
for cluster in range(best_k):
    print(f"\n**Cluster {cluster}: Unraveling the Patterns**\n")
    cluster_data_kmodes = dataset[dataset['Cluster_KModes'] == cluster]
    cluster_data_stratified_kmodes = dataset1_stratified.loc[cluster_data_kmodes.index]

    total_count = len(cluster_data_stratified_kmodes)  # Calculate total_count here

    if not cluster_data_stratified_kmodes.empty:
        for feature in dataset1_stratified.columns:  # Iterate through all features
            mode_value = cluster_data_stratified_kmodes[feature].mode()
            if not mode_value.empty:
                value = mode_value.iloc[0]  # Get the mode value
                count = cluster_data_stratified_kmodes[feature].value_counts().loc[value] # Get the count of the mode
                # If the value is encoded (bytes), decode it
                if isinstance(value, bytes):
                    value = value.decode('utf-8')
                print(f"* {feature}: {value}                [{count}/{total_count}]               (Most Frequent)")  # Highlight most frequent with count and total
            else:
                print(f"* {feature}: Diverse (No Clear Majority)")  # Indicate diversity
    else:
        print(f"Cluster {cluster}: This cluster is empty!")  # Inform about empty clusters

# --- KMeans ---
dataset['Cluster_KMeans'] = kmeans_labels  # Add KMeans cluster labels to dataset

print("\n--- KMeans Clustering Results ---\n")
for cluster in range(best_k_kmeans):  # Use best_k_kmeans for KMeans
    print(f"\n**Cluster {cluster}: Exploring the Data Landscape**\n")
    cluster_data_kmeans = dataset[dataset['Cluster_KMeans'] == cluster]
    cluster_data_stratified_kmeans = dataset1_stratified.loc[cluster_data_kmeans.index]

    total_count = len(cluster_data_stratified_kmeans)  # Calculate total_count here

    if not cluster_data_stratified_kmeans.empty:
        for feature in dataset1_stratified.columns:
            mode_value = cluster_data_stratified_kmeans[feature].mode()
            if not mode_value.empty:
                value = mode_value.iloc[0]
                count = cluster_data_stratified_kmeans[feature].value_counts().loc[value]
                if isinstance(value, bytes):
                    value = value.decode('utf-8')
                print(f"* {feature}: {value}                [{count}/{total_count}]                (Dominant)")  # Emphasize dominance with count and total and spacing
            else:
                print(f"* {feature}: Varied (No Single Prevailing Value)")  # Show variation
    else:
        print(f"Cluster {cluster}: This cluster has no members!")  # Explain empty clusters

# --- HDBSCAN ---
dataset['Cluster_HDBSCAN'] = hdbscan_labels  # Add HDBSCAN cluster labels to dataset
# Get unique cluster labels, excluding noise (-1)
hdbscan_clusters = np.unique(hdbscan_labels)
hdbscan_clusters = hdbscan_clusters[hdbscan_clusters != -1]

print("\n--- HDBSCAN Clustering Results ---\n")
for cluster in hdbscan_clusters:
    print(f"\n**Cluster {cluster}: Delving into Density-Based Groups**\n")
    cluster_data_hdbscan = dataset[dataset['Cluster_HDBSCAN'] == cluster]
    cluster_data_stratified_hdbscan = dataset1_stratified.loc[cluster_data_hdbscan.index]

    total_count = len(cluster_data_stratified_hdbscan)  # Calculate total_count here

    if not cluster_data_stratified_hdbscan.empty:
        mode_values = cluster_data_stratified_hdbscan.mode().iloc[0]
        for feature in dataset1_stratified.columns:
            if feature in mode_values:
                value = mode_values[feature]
                if isinstance(value, bytes):
                    value = value.decode('utf-8')
                print(f"* {feature}: {value}                [{count}/{total_count}]               (Mode of features)")  # Highlight most frequent with count and total
            else:
                print(f"* {feature}:                  Wide Range (No Central Value)")  # Highlight range
    else:
        print(f"Cluster {cluster}: This cluster lacks data points!")  # Describe empty clusters

# **Classification**

**Importing essential libraries for Classification**

In [ ]:
# Machine Learning Models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
# Evaluation Metrics
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve, confusion_matrix
# Warnings
import warnings
warnings.filterwarnings('ignore')
# Set Seaborn style for better visuals
sns.set_style("whitegrid")
print("Libraries successfully imported.")

**Loading the dataset**

In [ ]:
from pathlib import Path

PROCESSED_DATA_PATH = Path("outputs/cleaned_overdose_records.csv")
if not PROCESSED_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Processed dataset not found at {PROCESSED_DATA_PATH}. "
        "Run the preprocessing cells in this notebook first."
    )

# Load dataset into a DataFrame
df = pd.read_csv(PROCESSED_DATA_PATH)
# Display Basic Info
df.info()
df.head()

**Data Preprocessing**

In [ ]:
# Drop unnecessary columns if needed (e.g., "Date Type")
df.drop(columns=['Date Type'], inplace=True, errors='ignore')
# Define features (X) and target (y)
X = df.drop(columns=['Death City'])  # Features
y = (df['Death City'].isin(["HARTFORD", "NEW HAVEN", "BRIDGEPORT"])).astype(int)  # Example: Mark high-risk cities
# Split dataset into Train (80%) & Test (20%) before encoding
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# Encode categorical variables (only on training set to prevent data leakage)
label_encoders = {}
categorical_cols = ['Sex', 'Race', 'Residence City']
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])  # Apply same transformation to test set
    label_encoders[col] = le  # Store encoders for future inverse transformation if needed
# Standardize numerical features (optional but useful for SVM & ANN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data Preprocessing Completed!")
print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

**Implementing Classification Models**

In [ ]:
# Initialize models
models = {
    "Support Vector Machine": SVC(probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting Trees": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}
# Train and evaluate models
model_results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)  # Train model
    y_pred = model.predict(X_test_scaled)  # Predict
    y_proba = model.predict_proba(X_test_scaled)[:, 1]  # Get probabilities for AUC
    # Evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_proba)
    classification_rep = classification_report(y_test, y_pred)
    model_results[name] = {
        "Accuracy": accuracy,
        "AUC Score": auc_score,
        "Classification Report": classification_rep
    }
    # Print results
    print(f"\n=== {name} ===")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    print("Classification Report:\n", classification_rep)
print("\nModel Training Completed!")

**Implementing Artificial Neural Network (ANN)**

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
# Build ANN Model
ann_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification output
])
# Compile ANN Model
ann_model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
# Train the Model
history = ann_model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),
                        epochs=50, batch_size=32, verbose=1)
# Evaluate the Model
y_pred_ann = (ann_model.predict(X_test_scaled) > 0.5).astype(int)  # Convert probabilities to binary class
y_proba_ann = ann_model.predict(X_test_scaled).flatten()
# ANN Performance Metrics
accuracy_ann = accuracy_score(y_test, y_pred_ann)
auc_ann = roc_auc_score(y_test, y_proba_ann)
classification_rep_ann = classification_report(y_test, y_pred_ann)
print("\n=== Artificial Neural Network (ANN) ===")
print(f"Accuracy: {accuracy_ann:.4f}")
print(f"AUC Score: {auc_ann:.4f}")
print("Classification Report:\n", classification_rep_ann)

**Comapring all of the implemented models to determine the best model**

In [ ]:
from sklearn.metrics import roc_curve
# Store all model results (including ANN)
all_model_results = model_results.copy()
all_model_results["Artificial Neural Network"] = {
    "Accuracy": accuracy_ann,
    "AUC Score": auc_ann,
    "Classification Report": classification_rep_ann
}
# Extract Precision, Recall, and F1-score for all models
precision_scores, recall_scores, f1_scores = {}, {}, {}
for model, metrics in all_model_results.items():
    report_lines = metrics["Classification Report"].split("\n")
    avg_line = report_lines[-2].split()  # Extract 'weighted avg' line
    precision_scores[model] = float(avg_line[2])
    recall_scores[model] = float(avg_line[3])
    f1_scores[model] = float(avg_line[4])
# Convert results into DataFrame
model_metrics_df = pd.DataFrame(all_model_results).T
performance_df = pd.DataFrame({
    "Precision": precision_scores,
    "Recall": recall_scores,
    "F1-Score": f1_scores
})
# Set consistent figure size for all visualizations
figsize = (10, 6)
# === Accuracy & AUC Score Bar Plot ===
plt.figure(figsize=figsize)
ax = model_metrics_df[['Accuracy', 'AUC Score']].plot(kind='bar', colormap='viridis', alpha=0.85, figsize=figsize)
plt.title("Model Accuracy & AUC Score Comparison")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
# === Precision, Recall, and F1-Score Bar Plot ===
plt.figure(figsize=figsize)
ax = performance_df.plot(kind='bar', colormap='coolwarm', alpha=0.85, figsize=figsize)
plt.title("Model Precision, Recall, and F1-Score Comparison")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
# === ROC Curve for All Models ===
plt.figure(figsize=figsize)
for name, model in models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={model_results[name]['AUC Score']:.4f})")
# Add ANN to the ROC curve
fpr_ann, tpr_ann, _ = roc_curve(y_test, y_proba_ann)
plt.plot(fpr_ann, tpr_ann, color='blue', label=f"Artificial Neural Network (AUC={auc_ann:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison for All Models")
plt.legend()
plt.show()

**Improving Gradient Boosting Trees (GBT) model**

**Cross-validation**

In [ ]:
from sklearn.model_selection import cross_val_score
# Initialize GBT with default hyperparameters
gbt = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
# Perform 5-fold cross-validation
cv_scores = cross_val_score(gbt, X_train_scaled, y_train, cv=5, scoring='accuracy')
# Print results
print(f"Cross-validation Accuracy Scores: {cv_scores}")
print(f"Mean Accuracy: {cv_scores.mean():.4f}")
print(f"Standard Deviation: {cv_scores.std():.4f}")

**Hyperparameter tuning with GridSearchCV**

In [ ]:
from sklearn.model_selection import GridSearchCV
# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1],
    'max_depth': [3, 5],
    'subsample': [0.8, 1.0]
}
# Initialize GBT model
gbt = GradientBoostingClassifier(random_state=42)
# Perform Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(gbt, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=2)
grid_search.fit(X_train_scaled, y_train)
# Best hyperparameters
print(f"Best Hyperparameters: {grid_search.best_params_}")
# Train GBT with best hyperparameters
best_gbt = grid_search.best_estimator_
best_gbt.fit(X_train_scaled, y_train)
# Evaluate improved GBT model
y_pred_best_gbt = best_gbt.predict(X_test_scaled)
y_proba_best_gbt = best_gbt.predict_proba(X_test_scaled)[:, 1]
accuracy_best_gbt = accuracy_score(y_test, y_pred_best_gbt)
auc_best_gbt = roc_auc_score(y_test, y_proba_best_gbt)
classification_rep_best_gbt = classification_report(y_test, y_pred_best_gbt)
print("\n=== Improved Gradient Boosting Trees ===")
print(f"Accuracy: {accuracy_best_gbt:.4f}")
print(f"AUC Score: {auc_best_gbt:.4f}")
print("Classification Report:\n", classification_rep_best_gbt)

**Visualizations for GBT**

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve
# Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
# === Confusion Matrix ===
cm = confusion_matrix(y_test, y_pred_best_gbt)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low-Risk City', 'High-Risk City'],
            yticklabels=['Low-Risk City', 'High-Risk City'], ax=axes[0, 0])
axes[0, 0].set_title('Confusion Matrix - GBT')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_ylabel('Actual')
# === AUC-ROC Curve ===
fpr, tpr, _ = roc_curve(y_test, y_proba_best_gbt)
axes[0, 1].plot(fpr, tpr, color='blue', label=f"GBT (AUC = {auc_best_gbt:.4f})")
axes[0, 1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[0, 1].set_title('AUC-ROC Curve - GBT')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].legend()
# === Feature Importance Plot ===
feat_importances = pd.DataFrame({'Feature': X.columns, 'Importance': best_gbt.feature_importances_})
feat_importances = feat_importances.sort_values(by='Importance', ascending=False)
sns.barplot(x='Importance', y='Feature', data=feat_importances.head(10), palette='viridis', ax=axes[1, 0])
axes[1, 0].set_title("Top 10 Feature Importances - GBT")
axes[1, 0].set_xlabel("Importance")
axes[1, 0].set_ylabel("Feature")
# === Distribution of Predicted Probabilities ===
sns.histplot(y_proba_best_gbt, bins=30, kde=True, color='purple', ax=axes[1, 1])
axes[1, 1].set_title("Prediction Probability Distribution - GBT")
axes[1, 1].set_xlabel("Predicted Probability of High-Risk City")
axes[1, 1].set_ylabel("Count")
# Adjust layout
plt.tight_layout()
plt.show()

**SHAP Beeswarm Plot (Global Feature Importance)**

In [ ]:
import shap
# Initialize SHAP explainer
explainer = shap.Explainer(best_gbt, X_test_scaled)
shap_values = explainer(X_test_scaled)
# SHAP Beeswarm Plot
shap.summary_plot(shap_values, X_test)

**Precision-Recall Curve**

In [ ]:
from sklearn.metrics import precision_recall_curve
# Compute Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_proba_best_gbt)
# Plot
plt.figure(figsize=(8,6))
plt.plot(recall, precision, marker='.', label='GBT', color='purple', linewidth=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - GBT")
plt.legend()
plt.show()

**3D Feature Importance Scatter Plot**

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
# Select top 3 features
top_3_features = feat_importances.head(3)['Feature'].values
# 3D Scatter Plot
fig = plt.figure(figsize=(10,8))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(X_test[top_3_features[0]], X_test[top_3_features[1]], X_test[top_3_features[2]],
                c=y_proba_best_gbt, cmap='coolwarm', alpha=0.7)
ax.set_xlabel(top_3_features[0])
ax.set_ylabel(top_3_features[1])
ax.set_zlabel(top_3_features[2])
ax.set_title("3D Feature Importance Visualization - GBT")
plt.colorbar(sc, label="Predicted Probability of High-Risk City")
plt.show()

**Boxplots to Visualize Outliers**

In [ ]:
# Select numerical columns (Age + Drug-related features)
num_cols = ['Age'] + [col for col in df.columns if col not in ['Sex', 'Race', 'Residence City', 'Death City', 'Year']]
# Create Boxplots
plt.figure(figsize=(15, 8))
sns.boxplot(data=df[num_cols], orient='h', palette='coolwarm')
plt.title("Boxplot of Numerical Features (Detecting Outliers)", fontsize=14)
plt.xlabel("Value Distribution")
plt.show()

**Detecting Statistical Outliers using Z-score**

In [ ]:
from scipy.stats import zscore
# Compute Z-scores
z_scores = df[num_cols].apply(zscore)
# Count outliers (Z-score threshold > 3)
outlier_counts = (z_scores.abs() > 3).sum()
# Display
print("Outlier Counts per Feature (Z-score > 3):")
print(outlier_counts[outlier_counts > 0])

**IQR Method: Visualizing Outlier Distribution**

In [ ]:
# Compute IQR
Q1 = df[num_cols].quantile(0.25)
Q3 = df[num_cols].quantile(0.75)
IQR = Q3 - Q1
# Define outlier thresholds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
# Identify outliers
outliers = ((df[num_cols] < lower_bound) | (df[num_cols] > upper_bound)).sum()
# Plot Outlier Distribution
plt.figure(figsize=(12, 6))
sns.barplot(x=outliers.index, y=outliers.values, palette="coolwarm")
plt.xticks(rotation=90)
plt.title("Outlier Count per Feature (IQR Method)", fontsize=14)
plt.ylabel("Outlier Count")
plt.xlabel("Feature")
plt.show()

# **Regression**

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


# Load the dataset
df = df.copy()  # Using df.copy() to create a deep copy


# Define features and target
features = [
    'Age', 'Sex', 'Race', 'Residence City', 'Heroin', 'Cocaine', 'Fentanyl',
    'Oxycodone', 'Oxymorphone', 'Ethanol', 'Hydrocodone', 'Benzodiazepine',
    'Methadone', 'Meth/Amphetamine', 'Amphet', 'Tramad', 'Hydromorphone',
    'Morphine (Not Heroin)', 'Xylazine', 'Gabapentin', 'Year'
]
target = 'Death City'

# Drop rows with missing values in essential columns
df = df[features + [target]].dropna()

# Convert categorical variables
df['Sex'] = df['Sex'].astype(str)
df['Race'] = df['Race'].astype(str)
df['Residence City'] = df['Residence City'].astype(str)
df['Death City'] = df['Death City'].astype(str)

# Convert target variable into numerical counts per city
df[target] = df[target].map(df[target].value_counts())

# Define numerical and categorical features
numerical_features = ['Age', 'Year'] + [
    'Heroin', 'Cocaine', 'Fentanyl', 'Oxycodone', 'Oxymorphone', 'Ethanol',
    'Hydrocodone', 'Benzodiazepine', 'Methadone', 'Meth/Amphetamine', 'Amphet',
    'Tramad', 'Hydromorphone', 'Morphine (Not Heroin)', 'Xylazine', 'Gabapentin'
]
categorical_features = ['Sex', 'Race', 'Residence City']

# Split dataset into training and testing
X = df[features]
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



# Preprocessing: Scaling numerical data and encoding categorical data
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])



# Feature engineering: Adding polynomial features after splitting to avoid data leakage
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train[numerical_features])
X_test_poly = poly.transform(X_test[numerical_features])


# Define models with hyperparameter tuning
param_grid_elastic = {'alpha': [0.005, 0.0075, 0.01, 0.1], 'l1_ratio': [0.3, 0.5, 0.9]}  # alpha: Regularization strength, l1_ratio: Mixing parameter between L1 and L2 penalties
param_grid_ridge = {'alpha': [0.01, 0.1, 1.0, 10.0]}  # alpha: Regularization strength
param_grid_lasso = {'alpha': [0.01, 0.1, 1.0, 10.0]}  # alpha: Regularization strength
param_grid_rf = {'n_estimators': [50, 100, 200], 'max_depth': [5, 8, None]}  # n_estimators: Number of trees, max_depth: Maximum depth of each tree
param_grid_xgb = {
    'n_estimators': [75, 100, 125],
    'learning_rate': [0.125, 0.15, 0.175],
    'max_depth': [2, 5, 7]
}

models = {
    'ElasticNet': GridSearchCV(ElasticNet(random_state=42), param_grid_elastic, cv=4, scoring='r2'),
    'Ridge': GridSearchCV(Ridge(random_state=42), param_grid_ridge, cv=4, scoring='r2'),
    'Lasso': GridSearchCV(Lasso(random_state=42), param_grid_lasso, cv=4, scoring='r2'),
    'RandomForest': GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=4, scoring='r2'),
    'XGBoost': GridSearchCV(XGBRegressor(random_state=42), param_grid_xgb, cv=4, scoring='r2'),
}

# Train models and evaluate performance
results = {}
for name, model in models.items():

    # Fit preprocessor on the training data only
    X_train_preprocessed = preprocessor.fit_transform(X_train)
    X_test_preprocessed = preprocessor.transform(X_test)

    pipeline = Pipeline(steps=[('regressor', model)]) # Include preprocessor in pipeline

    start_time = time.time()  # Record start time
    pipeline.fit(X_train_preprocessed, y_train)  # Fit on preprocessed data
    end_time = time.time()  # Record end time

    training_time = end_time - start_time
    print(f"Training time for {name}: {training_time:.2f} seconds")

    start_time = time.time()  # Record start time for prediction
    y_pred = pipeline.predict(X_test_preprocessed)  # Predict on preprocessed data
    end_time = time.time()  # Record end time for prediction

    prediction_time = end_time - start_time
    print(f"Prediction time for {name}: {prediction_time:.2f} seconds")

    # Print best parameters
    print(f"Best parameters for {name}: {model.best_params_}\n")

    # Compute performance metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)

    results[name] = {'R² Score': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse}

# Stacking Ensemble
estimators = [
    ('RandomForest', Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(random_state=42))])),
    ('XGBoost', Pipeline(steps=[('preprocessor', preprocessor), ('regressor', XGBRegressor(random_state=42))])),
    ('Ridge', Pipeline(steps=[('preprocessor', preprocessor), ('regressor', Ridge(random_state=42))])),
]

stacking_regressor = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(random_state=42)
)

start_time = time.time()  # Record start time for stacking ensemble training
stacking_regressor.fit(X_train, y_train)
end_time = time.time()  # Record end time for stacking ensemble training

training_time = end_time - start_time
print(f"Training time for Stacking Ensemble: {training_time:.2f} seconds")

start_time = time.time()  # Record start time for stacking ensemble prediction
y_pred_stacking = stacking_regressor.predict(X_test)
end_time = time.time()  # Record end time for stacking ensemble prediction

prediction_time = end_time - start_time
print(f"Prediction time for Stacking Ensemble: {prediction_time:.2f} seconds\n")

# Convert results to DataFrame
results_df = pd.DataFrame(results).T
print(results_df)

# Evaluate stacking ensemble performance
r2_stacking = r2_score(y_test, y_pred_stacking)
mae_stacking = mean_absolute_error(y_test, y_pred_stacking)
mse_stacking = mean_squared_error(y_test, y_pred_stacking)
rmse_stacking = np.sqrt(mse_stacking)

print(f"\nStacking Ensemble - R² Score: {r2_stacking:.4f}")
print(f"Stacking Ensemble - MAE: {mae_stacking:.4f}")
print(f"Stacking Ensemble - MSE: {mse_stacking:.4f}")
print(f"Stacking Ensemble - RMSE: {rmse_stacking:.4f}\n")
# Add stacking ensemble results to the results dictionary
results['Stacking Ensemble'] = {'R² Score': r2_stacking, 'MAE': mae_stacking, 'MSE': mse_stacking, 'RMSE': rmse_stacking}

# Convert results to DataFrame
results_df = pd.DataFrame(results).T

# Visualization of model performance
plt.figure(figsize=(12, 8))
metrics = ['R² Score', 'MAE', 'MSE', 'RMSE']
y_ranges = [(0.71, 0.74), (46, 54), (8800, 9500), (92, 98)]  # Define y-axis ranges


for i, metric in enumerate(metrics):  # Use enumerate to get the index i
    plt.figure(figsize=(10, 6))
    sns.barplot(x=results_df.index, y=results_df[metric], palette='viridis')
    plt.title(f'Comparison of {metric} Across Models', fontsize=14)
    plt.xlabel('Models', fontsize=12)
    plt.ylabel(metric, fontsize=12)
    plt.xticks(rotation=45, fontsize=10)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()  # Adjust layout to prevent labels from overlapping
    plt.ylim(y_ranges[i])  # Set y-axis range for current metric
    plt.show()

# Create a dictionary to store prediction errors
errors = {}

# Get prediction errors for each model and store them in the dictionary
for name, model in models.items():
    y_pred = model.predict(preprocessor.transform(X_test))  # Preprocess before prediction
    errors[name] = y_test - y_pred  # Calculate prediction errors

# Add stacking ensemble errors to the dictionary
errors['Stacking Ensemble'] = y_test - y_pred_stacking

# Create a DataFrame from the errors dictionary
errors_df = pd.DataFrame(errors)

# Visualize prediction errors using a box plot with restricted range
plt.figure(figsize=(10, 6))
sns.boxplot(data=errors_df, palette='viridis')
plt.title('Distribution of Prediction Errors Across Models', fontsize=14)
plt.xlabel('Models', fontsize=12)
plt.ylabel('Prediction Error', fontsize=12)
plt.xticks(rotation=45, fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(-150, 150)  # Restrict y-axis range
plt.tight_layout()  # Adjust layout to prevent labels from overlapping
plt.show()